# Lab 1：我的第一個 LangGraph — HP 產品查詢 Agent（30 min）

## 目標
- 體驗 LangGraph 的核心元件：**StateGraph / Node / Edge / Conditional Edge**
- 建立一個能查詢 HP 產品規格的簡易 Agent
- 理解 Agent 如何「決定」要不要使用工具

## 架構
```
START → llm_node → [需要工具?] → tool_node → llm_node → END
                   [不需要]  → END
```

In [ ]:
%%capture
!pip install langgraph langchain-openai langchain-core

In [ ]:
import os
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# --- API Key 設定 ---
# 在 Colab 中可以用 Secrets 或直接貼上
# os.environ["OPENAI_API_KEY"] = "sk-..."

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Step 1: 定義 State

State 是 LangGraph 中所有節點共享的資料結構。每個節點讀取 State、處理後回傳更新。

In [ ]:
# 定義 Agent 的 State
class AgentState(TypedDict):
    question: str       # 使用者的問題
    tool_call: str      # 要呼叫的工具名稱（空字串表示不需要工具）
    tool_input: str     # 工具的輸入參數
    tool_output: str    # 工具的輸出結果
    answer: str         # 最終回答

## Step 2: 建立工具 — HP 產品規格查詢

這是一個模擬的產品資料庫，實際場景中可以接 API 或資料庫。

In [ ]:
# 模擬 HP 產品規格資料庫
HP_PRODUCTS = {
    "elitebook 855": {
        "名稱": "HP EliteBook 855 G10",
        "CPU": "AMD Ryzen 7 PRO 7840U",
        "RAM": "32GB DDR5",
        "儲存": "512GB PCIe NVMe SSD",
        "螢幕": "15.6 吋 FHD IPS",
        "重量": "1.74 kg",
    },
    "probook 450": {
        "名稱": "HP ProBook 450 G10",
        "CPU": "Intel Core i5-1335U",
        "RAM": "16GB DDR4",
        "儲存": "256GB PCIe NVMe SSD",
        "螢幕": "15.6 吋 FHD IPS",
        "重量": "1.79 kg",
    },
    "pavilion 15": {
        "名稱": "HP Pavilion 15-eg3000",
        "CPU": "Intel Core i7-1355U",
        "RAM": "16GB DDR4",
        "儲存": "1TB PCIe NVMe SSD",
        "螢幕": "15.6 吋 FHD IPS 觸控",
        "重量": "1.75 kg",
    },
}

def query_product_spec(product_name: str) -> str:
    """查詢 HP 產品規格"""
    product_name = product_name.lower().strip()
    for key, spec in HP_PRODUCTS.items():
        if key in product_name:
            lines = [f"{k}: {v}" for k, v in spec.items()]
            return "\n".join(lines)
    return f"找不到產品：{product_name}。目前支援的產品有：EliteBook 855、ProBook 450、Pavilion 15"

# 測試一下
print(query_product_spec("EliteBook 855"))

## Step 3: 建立 Node

我們需要兩個節點：
1. **llm_node** — 讓 LLM 決定要不要用工具，或直接回答
2. **tool_node** — 執行工具查詢

### TODO
- 在 `llm_node` 中，呼叫 LLM 並解析回應
- 在 `tool_node` 中，根據 `tool_call` 呼叫對應的工具

In [ ]:
def llm_node(state: AgentState) -> dict:
    """LLM 節點：判斷是否需要工具，或直接回答"""
    question = state["question"]
    tool_output = state.get("tool_output", "")

    # 如果已經有工具輸出，讓 LLM 根據工具結果回答
    if tool_output:
        prompt = f"""根據以下產品資訊回答使用者的問題。
產品資訊：
{tool_output}

使用者問題：{question}
請用繁體中文回答。"""

        # =============================================
        # TODO: 呼叫 LLM 取得回答（約 2 行）
        # 提示：用 llm.invoke() 並取出 .content
        # response = llm.invoke(___)
        # return {"answer": ___}
        # =============================================
        pass

    # 第一次進來，判斷需不需要工具
    prompt = f"""你是 HP 產品助理。使用者問了一個問題，請判斷：
- 如果需要查詢產品規格，回覆格式：TOOL:query_product_spec:產品名稱
- 如果不需要工具就能回答（例如打招呼），直接回覆答案

使用者問題：{question}"""

    # =============================================
    # TODO: 呼叫 LLM 取得回應（約 5 行）
    # 提示：
    # 1. response = llm.invoke(___)
    # 2. text = response.content
    # 3. 判斷 text 是否以 "TOOL:" 開頭
    #    - 是 → 解析出 tool_call 和 tool_input，回傳更新
    #    - 否 → 直接回傳 answer
    # =============================================
    pass


def tool_node(state: AgentState) -> dict:
    """工具節點：執行對應的工具"""
    tool_call = state["tool_call"]
    tool_input = state["tool_input"]

    # =============================================
    # TODO: 根據 tool_call 呼叫對應的工具（約 3 行）
    # 提示：
    # if tool_call == "query_product_spec":
    #     result = query_product_spec(___)
    # return {"tool_output": ___}
    # =============================================
    pass

## Step 4: 建立 Graph 並加入 Conditional Edge

Conditional Edge 讓 Agent 根據 State 動態決定下一步走向哪個節點。

### TODO
- 完成 `should_use_tool` 路由函數

In [ ]:
# =============================================
# TODO: 完成路由函數（約 3 行）
# 提示：檢查 state["tool_call"] 是否有值
#   - 有值 → 回傳 "use_tool"
#   - 沒有 → 回傳 "end"
# =============================================
def should_use_tool(state: AgentState) -> Literal["use_tool", "end"]:
    """路由函數：決定要不要走工具節點"""
    pass


# 建立 Graph
graph = StateGraph(AgentState)

# 加入節點
graph.add_node("llm_node", llm_node)
graph.add_node("tool_node", tool_node)

# 設定起點
graph.set_entry_point("llm_node")

# 加入 Conditional Edge：llm_node 之後根據條件走不同路
graph.add_conditional_edges(
    "llm_node",
    should_use_tool,
    {
        "use_tool": "tool_node",
        "end": END,
    }
)

# tool_node 完成後回到 llm_node（讓 LLM 根據工具結果回答）
graph.add_edge("tool_node", "llm_node")

# 編譯 Graph
app = graph.compile()
print("Graph 編譯完成！")

## Step 5: 執行測試

In [ ]:
# 測試 1：需要工具的問題
print("=" * 50)
print("測試 1：產品規格查詢")
print("=" * 50)
result = app.invoke({
    "question": "EliteBook 855 有多少 RAM？",
    "tool_call": "",
    "tool_input": "",
    "tool_output": "",
    "answer": "",
})
print(f"問題：{result['question']}")
print(f"工具呼叫：{result['tool_call']}")
print(f"回答：{result['answer']}")

print()

# 測試 2：不需要工具的問題
print("=" * 50)
print("測試 2：一般對話")
print("=" * 50)
result2 = app.invoke({
    "question": "你好",
    "tool_call": "",
    "tool_input": "",
    "tool_output": "",
    "answer": "",
})
print(f"問題：{result2['question']}")
print(f"回答：{result2['answer']}")

In [ ]:
# 畫出 Graph 結構圖
from IPython.display import Image, display

try:
    img = app.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    print(f"畫圖需要額外套件，可忽略：{e}")
    # 文字版
    print(app.get_graph().draw_mermaid())

## 延伸挑戰

試著加入第二個工具 `check_inventory`，讓 Agent 能回答「EliteBook 855 還有庫存嗎？」

提示：
1. 新增一個 `check_inventory` 函數
2. 修改 `llm_node` 的 prompt，讓 LLM 知道有兩個工具可以用
3. 修改 `tool_node`，加入新工具的分支

```python
HP_INVENTORY = {
    "elitebook 855": {"庫存": 15, "倉庫": "台北"},
    "probook 450": {"庫存": 0, "倉庫": "缺貨中"},
    "pavilion 15": {"庫存": 42, "倉庫": "高雄"},
}
```